In [2]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [3]:
import requests
import numpy as np
import faiss
import torch
import random
from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

W0705 09:33:00.742000 61626 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [4]:
response = requests.get('https://raw.githubusercontent.com/run-llama/llama_index/main/docs/docs/examples/data/paul_graham/paul_graham_essay.txt')
text = response.text

In [5]:
print(text[700: 1000])

e a mini Bond villain's lair down there, with all these alien-looking machines — CPU, disk drives, printer, card reader — sitting up on a raised floor under bright fluorescent lights.

The language we used was an early version of Fortran. You had to type programs on punch cards, then stack them in t


In [6]:
CHUNK_SIZE = 1400    # Size of each chunk
CHUNK_OVERLAP = 100

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ",],
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP

)

In [8]:
# chunks = [text[i - CHUNK_OVERLAP:i + CHUNK_SIZE] for i in range(0, len(text), CHUNK_SIZE)]
chunks = text_splitter.split_text(text)
print(len(chunks))

65


In [9]:
for sample_text in random.sample(chunks, 3):  # Display 3 random chunks
        print(sample_text[:500])  # Print the first 200 characters of each chunk
        print("")

What should I do next? Rtm's advice hadn't included anything about that. I wanted to do something completely different, so I decided I'd paint. I wanted to see how good I could get if I really focused on it. So the day after I stopped working on YC, I started painting. I was rusty and it took a while to get back into shape, but it was at least completely engaging. [18]

I spent most of the rest of 2014 painting. I'd never been able to work so uninterruptedly before, and I got to be better than I

I got so excited about this idea that I couldn't think about anything else. It seemed obvious that this was the future. I didn't particularly want to start another company, but it was clear that this idea would have to be embodied as one, so I decided to move to Cambridge and start it. I hoped to lure Robert into working on it with me, but there I ran into a hitch. Robert was now a postdoc at MIT, and though he'd made a lot of money the last time I'd lured him into working on one of my schemes

In [10]:
# use sentence transformer model with large max_seq_length
# you can use any model, but this one is known to work well with long texts
embeddings_model = SentenceTransformer('all-MiniLM-L6-v2')
print(embeddings_model.max_seq_length)
embeddings = np.array([embeddings_model.encode(chunk, normalize=True) for chunk in chunks])

256


In [11]:
# lets create a FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension, )
index.add(embeddings)

In [12]:
# lets test the index
question = "Did Yahoo stock go up?"
question_embeddings = np.array([embeddings_model.encode(question, normalize=True)])
distances, indices = index.search(question_embeddings, k=5, )

for i, idx in enumerate(indices[0]):
    print(f"Chunk {i+1} (Distance: {distances[0][i]}):")
    print(chunks[idx][:200])  # Print the first 200 characters of each chunk
    print()  # Add a newline for better readability

Chunk 1 (Distance: 1.0068285465240479):
Alas I hired lots more people, partly because our investors wanted me to, and partly because that's what startups did during the Internet Bubble. A company with just a handful of employees would have 

Chunk 2 (Distance: 1.0785348415374756):
The next year, from the summer of 1998 to the summer of 1999, must have been the least productive of my life. I didn't realize it at the time, but I was worn out from the effort and stress of running 

Chunk 3 (Distance: 1.3339022397994995):
We invited about 20 of the 225 groups to interview in person, and from those we picked 8 to fund. They were an impressive group. That first batch included reddit, Justin Kan and Emmett Shear, who went

Chunk 4 (Distance: 1.3998780250549316):
So I gave this talk, in the course of which I told them that the best sources of seed funding were successful startup founders, because then they'd be sources of advice too. Whereupon it seemed they w

Chunk 5 (Distance: 1.51440596580

In [13]:
retrieved_chunk = [chunks[i] for i in indices.tolist()[0]]
print("Retrieved Chunks:")
for chunk in retrieved_chunk:
    print(chunk[:200])  # Print the first 200 characters of each chunk
    print()  # Add a newline for better readability

Retrieved Chunks:
Alas I hired lots more people, partly because our investors wanted me to, and partly because that's what startups did during the Internet Bubble. A company with just a handful of employees would have 

The next year, from the summer of 1998 to the summer of 1999, must have been the least productive of my life. I didn't realize it at the time, but I was worn out from the effort and stress of running 

We invited about 20 of the 225 groups to interview in person, and from those we picked 8 to fund. They were an impressive group. That first batch included reddit, Justin Kan and Emmett Shear, who went

So I gave this talk, in the course of which I told them that the best sources of seed funding were successful startup founders, because then they'd be sources of advice too. Whereupon it seemed they w

Once again, ignorance worked in our favor. We had no idea how to be angel investors, and in Boston in 2005 there were no Ron Conways to learn from. So we just made what seeme

In [14]:
prompt = f"""
Context information is below.
---------------------
{"\n".join(retrieved_chunk)}
---------------------
Given the context information, answer the query.
Query: {question}
Answer:
"""

In [15]:
model_name = "google/gemma-3-1b-it"
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

In [16]:
model.get_memory_footprint()/1024**3  # in GB

1.862433673813939

In [17]:
input_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.2, top_p=0.95)
# only include the answer part of the output
# Extract only the newly generated tokens (exclude the input prompt)
answer_tokens = outputs[0][len(inputs.input_ids[0]):]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=True)
print("Answer:", answer)

Answer: Yes, Yahoo stock went up 5x in the next year after being bought by Yahoo.<end_of_turn>


256